# develop code


In [36]:
from langchain.agents import create_agent

In [37]:
import operator
from typing_extensions import TypedDict, List, Optional, Annotated
from pydantic import BaseModel, Field
from langgraph.graph import START, END, StateGraph
from langgraph.types import Send
from langchain_ollama.chat_models import ChatOllama
from langchain.messages import SystemMessage, HumanMessage
from uuid import UUID, uuid4


from dotenv import load_dotenv
load_dotenv()

True

In [38]:
class Task(BaseModel):
    
    id: UUID = Field( default_factory= uuid4)
    title: str
    brief: str = Field(..., description="Instructions on what to cover")


In [39]:
class Plan(BaseModel):
    
    blog_title: str
    tasks: List[Task]
     

In [40]:
class State(StateGraph):
    
    topic: str
    plan: Plan
    
    sections: Annotated[List[str], operator.add]
    final: str

In [44]:
llm = ChatOllama(model="qwen3:8b")

In [45]:
llm.invoke("hi")

AIMessage(content="Hello! 😊 How can I assist you today? Whether you have questions, need help with something, or just want to chat, I'm here for you! What's on your mind?", additional_kwargs={}, response_metadata={'model': 'qwen3:8b', 'created_at': '2026-02-10T06:23:12.2591855Z', 'done': True, 'done_reason': 'stop', 'total_duration': 7514988000, 'load_duration': 96027400, 'prompt_eval_count': 11, 'prompt_eval_duration': 299508100, 'eval_count': 146, 'eval_duration': 7087026000, 'logprobs': None, 'model_name': 'qwen3:8b', 'model_provider': 'ollama'}, id='lc_run--019c4637-c027-7e53-b27e-ddc685d11f1f-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 11, 'output_tokens': 146, 'total_tokens': 157})

In [46]:
def orchrestrator(state: State)-> dict:
    
    plan = llm.with_structured_output(Plan).invoke(
        [SystemMessage(content="""You are an expert editor that creates plan/steps for a blog based on the topic provided by the user"""),
         HumanMessage(content=f"Topic: {state['topic']}")]
    )
    return{"plan": plan}
    

In [47]:
def fanout(state: State):
    return[Send("worker", {"task": task, "topic": state["topic"], "plan":state["plan"]}) for task in state["plan"].tasks]

In [50]:
def worker(payload: dict)-> dict:
    task = payload["task"]
    topic = payload["topic"]
    plan = payload["plan"]
    blog_title = plan.blog_title
    
    section_md = llm.invoke([
        SystemMessage(content="write one clean markdown section"),
        HumanMessage(content=f"""
                     Blog: {blog_title},
                     Topic: {topic},
                     Section: {task.title}
                     Brief: {task.brief}
                     
                     Return only the section content in markdown.
                     """)
    ]).content.strip()
    
    return {"sections": [section_md]}    